# Unidad 9: Spark en arquitecturas de datos modernas

**Tecnicatura en Datos – Apache Spark (Databricks)**  
Notebook de práctica — `spark` ya está disponible en el entorno

---

## Contenido

1. ETL vs ELT: diferencias y cuándo usar cada uno
2. Data Lake, Data Warehouse y Data Lakehouse
3. Medallion Architecture (Bronze → Silver → Gold)
4. Patrones Lambda y Kappa (conceptual)
5. Orquestación: concepto de DAG con Airflow
6. Monitoreo: acumuladores, logging y métricas
7. Buenas prácticas en producción
8. Práctica guiada: pipeline con validación y reporte

## Configuración

In [ ]:
sc = spark.sparkContext
print(f"Spark version: {spark.version}")
print(f"App name:      {sc.appName}")

---
## 1. ETL vs ELT

### Ejemplo 1 — Simple: implementar un pipeline ETL con funciones modulares

In [ ]:
from pyspark.sql.functions import col, when, current_timestamp
from pyspark.sql import DataFrame

# --- EXTRACT ---
def extract_ventas() -> DataFrame:
    """Extrae datos crudos de ventas (simula lectura desde una fuente)."""
    data = [
        (1, "Ana",     1500.0, "AR", "2024-01-10", "web"),
        (2, "Luis",    -50.0,  "MX", "2024-01-11", "mobile"),  # monto negativo
        (3, "Marta",  2200.0,  "CL", "2024-01-12", "tienda"),
        (4, "Pedro",    None,  "AR", "2024-01-13", "web"),      # monto nulo
        (5, "Sofía",   800.0,  "MX", "2024-01-14", "mobile"),
        (6, "Diego",  3500.0,  "XX", "2024-01-15", "web"),      # país inválido
        (7, "Laura",  1200.0,  "BR", "2024-01-16", "tienda"),
    ]
    return spark.createDataFrame(data, ["id", "cliente", "monto", "pais", "fecha", "canal"])

# --- TRANSFORM ---
def transform_ventas(df: DataFrame) -> tuple:
    """Limpia y enriquece. Retorna (df_valido, df_rechazado)."""
    paises_validos = ["AR", "MX", "CL", "BR", "CO"]
    cond_valida = (
        col("monto").isNotNull() &
        (col("monto") > 0) &
        col("pais").isin(paises_validos)
    )
    df_valido = (df.filter(cond_valida)
        .withColumn("categoria",
            when(col("monto") < 1000, "bajo")
            .when(col("monto") < 3000, "medio")
            .otherwise("alto"))
        .withColumn("procesado_en", current_timestamp())
    )
    df_rechazado = df.filter(~cond_valida)
    return df_valido, df_rechazado

# --- LOAD ---
def load_ventas(df_valido: DataFrame, df_rechazado: DataFrame, ruta_base: str):
    """Escribe resultados en rutas separadas."""
    df_valido.write.mode("overwrite").parquet(f"{ruta_base}/validos/")
    df_rechazado.write.mode("overwrite").parquet(f"{ruta_base}/rechazados/")
    print(f"Válidos escritos:    {df_valido.count()} filas")
    print(f"Rechazados escritos: {df_rechazado.count()} filas")

# --- EJECUTAR PIPELINE ---
df_raw = extract_ventas()
df_valido, df_rechazado = transform_ventas(df_raw)
load_ventas(df_valido, df_rechazado, "/tmp/u09_etl/")

print("\nDatos válidos:")
df_valido.show()
print("Datos rechazados:")
df_rechazado.show()

**Resultado esperado:**
```
Válidos escritos:    4 filas
Rechazados escritos: 3 filas

Datos válidos:
+---+-------+------+----+----------+------+---------+-------------------+
| id|cliente| monto|pais|     fecha| canal|categoria|       procesado_en|
+---+-------+------+----+----------+------+---------+-------------------+
|  1|    Ana|1500.0|  AR|2024-01-10|   web|    medio|2024-...
|  3|  Marta|2200.0|  CL|2024-01-12|tienda|    medio|2024-...
|  5|  Sofía| 800.0|  MX|2024-01-14|mobile|     bajo|2024-...
|  7|  Laura|1200.0|  BR|2024-01-16|tienda|    medio|2024-...

Datos rechazados:    (Luis: monto negativo, Pedro: nulo, Diego: país XX)
```

---
## 2. Medallion Architecture: Bronze → Silver → Gold

### Ejemplo 2 — Medio: implementar las tres capas

```
Fuente → Bronze (raw) → Silver (limpio) → Gold (analítico)
```

In [ ]:
from pyspark.sql.functions import col, when, current_timestamp, lit, sum as spark_sum, avg, count
from pyspark.sql.functions import year, month, round as spark_round

# ===== BRONZE: datos crudos + metadata de ingesta =====
def crear_bronze():
    data = [(i, f"cli_{i%10}", float(i*150) if i%7!=0 else None,
             ["AR","MX","CL","BR"][i%4],
             f"2024-{(i%12)+1:02d}-{(i%28)+1:02d}",
             ["web","mobile","tienda"][i%3])
            for i in range(1, 51)]
    
    df_bronze = (spark.createDataFrame(data, ["id","cliente","monto","pais","fecha","canal"])
        .withColumn("_ingesta_ts", current_timestamp())
        .withColumn("_fuente",     lit("sistema_ventas"))
    )
    df_bronze.write.format("delta").mode("overwrite").save("/tmp/u09_bronze/")
    print(f"Bronze: {df_bronze.count()} filas ingresadas")
    return df_bronze

# ===== SILVER: limpieza, validación, tipos correctos =====
def bronze_a_silver():
    df_b = spark.read.format("delta").load("/tmp/u09_bronze/")
    
    cond_valida = col("monto").isNotNull() & (col("monto") > 0)
    df_s = (df_b
        .filter(cond_valida)
        .withColumn("año", year(col("fecha").cast("date")))
        .withColumn("mes", month(col("fecha").cast("date")))
        .withColumn("monto_total", spark_round(col("monto") * 1.21, 2))  # con IVA
        .drop("_ingesta_ts", "_fuente")
    )
    df_s.write.format("delta").mode("overwrite").partitionBy("año","mes").save("/tmp/u09_silver/")
    invalidas = df_b.count() - df_s.count()
    print(f"Silver: {df_s.count()} filas válidas, {invalidas} descartadas")
    return df_s

# ===== GOLD: agregaciones para BI =====
def silver_a_gold():
    df_s = spark.read.format("delta").load("/tmp/u09_silver/")
    
    df_g = (df_s
        .groupBy("pais", "canal", "año", "mes")
        .agg(
            count("id").alias("num_ventas"),
            spark_round(spark_sum("monto_total"), 2).alias("revenue"),
            spark_round(avg("monto"), 2).alias("ticket_promedio")
        )
        .orderBy("pais", "año", "mes")
    )
    df_g.write.format("delta").mode("overwrite").save("/tmp/u09_gold/")
    print(f"Gold: {df_g.count()} filas de resumen")
    return df_g

# Ejecutar pipeline completo
print("=== Pipeline Medallion ===")
crear_bronze()
bronze_a_silver()
df_gold = silver_a_gold()

print("\n=== Resultado Gold ===")
df_gold.show(12)

**Resultado esperado:**
```
=== Pipeline Medallion ===
Bronze: 50 filas ingresadas
Silver: 43 filas válidas, 7 descartadas
Gold: 12 filas de resumen

=== Resultado Gold ===
+----+------+----+---+----------+--------+---------------+
|pais| canal| año|mes|num_ventas| revenue|ticket_promedio|
+----+------+----+---+----------+--------+---------------+
|  AR|mobile|2024|  1|         2|  726.0 |          300.0|
|  AR|tienda|2024|  2|         1|  907.5 |          750.0|
...
```

---
## 3. Monitoreo con acumuladores y logging

### Ejemplo 3 — Avanzado: pipeline con métricas de calidad y logging estructurado

In [ ]:
import logging
import time
from pyspark.sql.functions import col, when, current_timestamp, udf
from pyspark.sql.types import StringType

# Configurar logger
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
log = logging.getLogger("pipeline.ventas")

# Acumuladores para métricas distribuidas
filas_ok      = sc.accumulator(0)
filas_nulas   = sc.accumulator(0)
filas_negativ = sc.accumulator(0)

@udf(StringType())
def categorizar_con_metricas(monto):
    """UDF que clasifica y acumula métricas."""
    if monto is None:
        filas_nulas.add(1)
        return "error_nulo"
    elif monto < 0:
        filas_negativ.add(1)
        return "error_negativo"
    else:
        filas_ok.add(1)
        if monto < 500:   return "bajo"
        elif monto < 2000: return "medio"
        else:              return "alto"

# Datos con errores
data_con_errores = [
    (1, 1500.0), (2, None), (3, -200.0), (4, 800.0),
    (5, 3000.0), (6, None), (7, 200.0),  (8, -50.0),
    (9, 4500.0), (10, 750.0)
]
df_test = spark.createDataFrame(data_con_errores, ["id", "monto"])

log.info(f"Iniciando pipeline. Filas de entrada: {df_test.count()}")
t_inicio = time.time()

# Aplicar UDF con métricas
df_procesado = df_test.withColumn("categoria", categorizar_con_metricas(col("monto")))
df_procesado.collect()   # trigger para ejecutar el UDF

duracion = time.time() - t_inicio

log.info(f"Pipeline completado en {duracion:.2f}s")

print("\n=== Métricas de calidad ===")
total = df_test.count()
print(f"Total filas:        {total}")
print(f"Válidas (ok):       {filas_ok.value}       ({filas_ok.value/total*100:.1f}%)")
print(f
print(f"Negativas:          {filas_negativ.value}       ({filas_negativ.value/total*100:.1f}%)")

print("\nResultado:")
df_procesado.show()

**Resultado esperado:**
```
2024-01-15 10:00:00 [INFO] Iniciando pipeline. Filas de entrada: 10
2024-01-15 10:00:01 [INFO] Pipeline completado en 1.23s

=== Métricas de calidad ===
Total filas:        10
Válidas (ok):       6       (60.0%)
Nulas:              2       (20.0%)
Negativas:          2       (20.0%)

+---+------+--------------+
| id| monto|     categoria|
+---+------+--------------+
|  1|1500.0|         medio|
|  2|  null|    error_nulo|
|  3|-200.0|error_negativo|
...
```

---
## 4. Concepto de orquestación con Airflow

Airflow orquesta pipelines mediante **DAGs** (Directed Acyclic Graphs). Este notebook muestra el concepto implementado en Python puro (sin Airflow), para entender la estructura de dependencias.

In [ ]:
import time
from datetime import datetime
from pyspark.sql.functions import col, current_timestamp

# Simular un DAG de Airflow con funciones Python
# En producción real: cada función sería un Task de Airflow

class TareaSimulada:
    def __init__(self, nombre):
        self.nombre    = nombre
        self.estado    = "pendiente"
        self.inicio    = None
        self.fin       = None
    
    def ejecutar(self, func, *args, **kwargs):
        self.estado = "ejecutando"
        self.inicio = datetime.now()
        print(f"[{self.inicio.strftime('%H:%M:%S')}] ▶ {self.nombre}")
        try:
            resultado = func(*args, **kwargs)
            self.estado = "completado"
            self.fin = datetime.now()
            dur = (self.fin - self.inicio).total_seconds()
            print(f"  ✓ {self.nombre} completado en {dur:.2f}s")
            return resultado
        except Exception as e:
            self.estado = "fallido"
            print(f"  ✗ {self.nombre} FALLÓ: {e}")
            raise

# Definir las tareas del pipeline
def tarea_extraer():
    data = [(i, f"prod_{i%5}", float(i*200), ["AR","MX"][i%2]) for i in range(1, 31)]
    return spark.createDataFrame(data, ["id", "producto", "monto", "pais"])

def tarea_limpiar(df):
    return df.filter(col("monto") > 0).withColumn("procesado_en", current_timestamp())

def tarea_agregar(df):
    return df.groupBy("pais", "producto").agg({"monto": "sum", "id": "count"})

def tarea_cargar(df):
    df.write.mode("overwrite").parquet("/tmp/u09_pipeline_output/")
    return df.count()

# Ejecutar DAG (en Airflow: las dependencias definen el orden)
print("=== Ejecutando DAG: pipeline_ventas ===")

t1 = TareaSimulada("extraer_datos")
t2 = TareaSimulada("limpiar_datos")
t3 = TareaSimulada("agregar_por_pais")
t4 = TareaSimulada("cargar_resultados")

# Dependencias: t1 >> t2 >> t3 >> t4
df_raw    = t1.ejecutar(tarea_extraer)
df_limpio = t2.ejecutar(tarea_limpiar, df_raw)
df_agg    = t3.ejecutar(tarea_agregar, df_limpio)
n_filas   = t4.ejecutar(tarea_cargar, df_agg)

print(f"\n=== Resumen del DAG ===")
for t in [t1, t2, t3, t4]:
    print(f"  {t.nombre:25} → {t.estado}")

print(f"\nFilas en resultado: {n_filas}")
spark.read.parquet("/tmp/u09_pipeline_output/").show()

**Resultado esperado:**
```
=== Ejecutando DAG: pipeline_ventas ===
[10:00:01] ▶ extraer_datos
  ✓ extraer_datos completado en 0.85s
[10:00:02] ▶ limpiar_datos
  ✓ limpiar_datos completado en 0.23s
[10:00:02] ▶ agregar_por_pais
  ✓ agregar_por_pais completado en 0.31s
[10:00:02] ▶ cargar_resultados
  ✓ cargar_resultados completado en 1.12s

=== Resumen del DAG ===
  extraer_datos             → completado
  limpiar_datos             → completado
  agregar_por_pais          → completado
  cargar_resultados         → completado

Filas en resultado: 10
+----+--------+---------+--------+
|pais|producto|sum(monto)|count(id)|
+----+--------+---------+--------+
|  AR|  prod_0|   3000.0 |       3|
...
```

---
## 5. Práctica guiada: pipeline completo con validación y reporte de calidad

In [ ]:
from pyspark.sql.functions import (col, when, current_timestamp, lit,
                                    sum as spark_sum, avg, count, round as spark_round,
                                    year, month)

# --- GENERAR DATOS CON ERRORES ---
import random
random.seed(42)

data_practica = []
paises = ["AR", "MX", "CL", "BR", "XX"]  # XX es inválido
canales = ["web", "mobile", "tienda"]
for i in range(1, 201):
    monto = float(i * 53.7) if i % 9 != 0 else None   # ~11% nulos
    if i % 15 == 0: monto = -100.0                     # ~7% negativos
    pais  = paises[i % 5]
    canal = canales[i % 3]
    fecha = f"2024-{(i%12)+1:02d}-{(i%28)+1:02d}"
    data_practica.append((i, f"cli_{i%20}", monto, pais, fecha, canal))

df_raw_p = spark.createDataFrame(data_practica, ["id","cliente","monto","pais","fecha","canal"])
print(f"Raw: {df_raw_p.count()} filas")

# --- VALIDACIÓN ---
paises_validos = ["AR", "MX", "CL", "BR", "CO"]
cond_ok = (
    col("monto").isNotNull() &
    (col("monto") > 0) &
    col("pais").isin(paises_validos)
)

df_valido_p   = df_raw_p.filter(cond_ok)
df_invalido_p = df_raw_p.filter(~cond_ok).withColumn(
    "motivo",
    when(col("monto").isNull(),       "monto_nulo")
    .when(col("monto") <= 0,          "monto_invalido")
    .when(~col("pais").isin(paises_validos), "pais_invalido")
    .otherwise("otro")
)

# --- REPORTE DE CALIDAD ---
total     = df_raw_p.count()
validos   = df_valido_p.count()
invalidos = df_invalido_p.count()

print("\n========== REPORTE DE CALIDAD ==========")
print(f" Total filas:             {total}")
print(f" Filas válidas:           {validos}  ({validos/total*100:.1f}%)")
print(f" Filas inválidas:         {invalidos}  ({invalidos/total*100:.1f}%)")
print("")
print(" Distribución de rechazos:")
df_invalido_p.groupBy("motivo").count().show()

# --- SILVER: enriquecer válidos ---
df_silver_p = (df_valido_p
    .withColumn("monto_con_iva", spark_round(col("monto") * 1.21, 2))
    .withColumn("categoria",
        when(col("monto") < 2000, "bajo")
        .when(col("monto") < 6000, "medio")
        .otherwise("alto"))
    .withColumn("año", year(col("fecha").cast("date")))
    .withColumn("mes", month(col("fecha").cast("date")))
)

# --- GOLD: resumen por canal y país ---
df_gold_p = (df_silver_p
    .groupBy("pais", "canal", "año", "mes")
    .agg(
        count("id").alias("num_ventas"),
        spark_round(spark_sum("monto_con_iva"), 2).alias("revenue_iva"),
        spark_round(avg("monto"), 2).alias("ticket_prom")
    )
    .orderBy("revenue_iva", ascending=False)
)

print("========== TOP 10 REVENUE POR CANAL/PAÍS ==========")
df_gold_p.show(10)

**Resultado esperado:**
```
Raw: 200 filas

========== REPORTE DE CALIDAD ==========
 Total filas:             200
 Filas válidas:           157  (78.5%)
 Filas inválidas:         43   (21.5%)

 Distribución de rechazos:
+---------------+-----+
|         motivo|count|
+---------------+-----+
|     monto_nulo|   22|
|  monto_invalido|   13|
|   pais_invalido|   40|
+---------------+-----+

========== TOP 10 REVENUE POR CANAL/PAÍS ==========
+----+------+----+---+----------+----------+-----------+
|pais| canal| año|mes|num_ventas| revenue_iva|ticket_prom|
+----+------+----+---+----------+----------+-----------+
|  AR|   web|2024|  1|        10|  65432.10|     5408.5|
...
```

---
## 6. Checklist de buenas prácticas en producción

```
☐ Schema explícito en lectura (no inferSchema=True)
☐ Checkpoint definido para streams
☐ Secrets manejados con Databricks Secrets o Vault
☐ Particionamiento apropiado para el volumen de datos
☐ Logging estructurado en todos los pasos clave
☐ Reintentos configurados en el orquestador
☐ Monitoreo de métricas: duración, filas procesadas, errores
☐ Pruebas con datos de muestra antes de producción
☐ Documentación del job: entradas, salidas, dependencias
☐ Datos rechazados guardados para auditoría
```

---
## Preguntas de revisión

1. ¿Cuál es la diferencia entre ETL y ELT? ¿Cuándo conviene cada uno?
2. ¿Qué problema resuelve Delta Lake que un Data Lake tradicional no puede?
3. ¿Cuáles son las tres capas de la Medallion Architecture y qué contiene cada una?
4. ¿Cuál es la principal desventaja de la arquitectura Lambda?
5. ¿Para qué sirven los acumuladores de Spark?
6. ¿Por qué es importante guardar los datos rechazados en un pipeline ETL?